In [54]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [55]:
import pandas as pd
import zipfile
import numpy as np
import re
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [56]:
zip_path = "archive.zip"  

with zipfile.ZipFile(zip_path) as z:
    fake = pd.read_csv(z.open("Fake.csv"))
    true = pd.read_csv(z.open("True.csv"))

In [57]:
fake["label"]=1
true["label"]=0

In [58]:
df=pd.concat([fake,true],axis=0)

In [59]:
df=df.sample(frac=1).reset_index(drop=True)

In [60]:
df.shape

(44898, 5)

In [61]:
df.head()

,title,text,subject,date,label
0,GREAT NEWS! House GOP Moderates Threaten “You ...,It looks like there is more than one good reas...,politics,"Oct 21, 2015",1
1,Kurdistan never intended to engage in war with...,CAIRO (Reuters) - The Kurdistan Regional Gove...,worldnews,"October 19, 2017",0
2,WOW! RUSSIAN HACKER Gives Stunning Details Of ...,A Russian man wanted by the Justice Department...,left-news,"May 12, 2017",1
3,Trump says will renegotiate NAFTA at 'appropri...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"January 23, 2017",0
4,CNN Host STUNNED As Trump Supporter Says Vote...,As Donald Trump and his supporters continue to...,News,"October 18, 2016",1


In [62]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [63]:
df['content']=df['text']+' '+df['title']

In [64]:
df['content']=df['content'].fillna('')

In [65]:
X=df.drop(columns='label',axis=1)
Y=df['label']

In [66]:
print(X)
print(Y)

                                                   title  ...                                            content
0      GREAT NEWS! House GOP Moderates Threaten “You ...  ...  It looks like there is more than one good reas...
1      Kurdistan never intended to engage in war with...  ...  CAIRO (Reuters) - The  Kurdistan Regional Gove...
2      WOW! RUSSIAN HACKER Gives Stunning Details Of ...  ...  A Russian man wanted by the Justice Department...
3      Trump says will renegotiate NAFTA at 'appropri...  ...  WASHINGTON (Reuters) - President Donald Trump ...
4       CNN Host STUNNED As Trump Supporter Says Vote...  ...  As Donald Trump and his supporters continue to...
...                                                  ...  ...                                                ...
44893  German police say Potsdam explosive package no...  ...  BERLIN (Reuters) - German authorities investig...
44894   Gender Equality Justice For Unwed American Mo...  ...  The U.S. Supreme Court has struck

In [72]:
def clean_text(content):
    content = re.sub('[^a-zA-Z]', ' ', content).lower().split()
    return ' '.join(word for word in content if word not in stop_words)

In [74]:
 df['content']=df['content'].apply(clean_text)

In [75]:
print(df['content'])

0        looks like one good reason elect paul ryan nex...
1        cairo reuters kurdistan regional government kr...
2        russian man wanted justice department charges ...
3        washington reuters president donald trump mond...
4        donald trump supporters continue live frighten...
                               ...                        
44893    berlin reuters german authorities investigatin...
44894    u supreme court struck backward gender discrim...
44895    trump peaking right time trump held rally st a...
44896    mooch asked get weak kneed husband christmas g...
44897    president first lady touched warsaw poland tod...
Name: content, Length: 44898, dtype: object


In [76]:
X=df['content'].values
Y=df['label'].values

In [77]:
print(X)
print(Y)

['looks like one good reason elect paul ryan next speaker house best news heard week moderate wing gop concerned house cannot coalesce behind rep paul ryan r wi someone like speaker pragmatic members caucus retire national journal reported monday de pend ing shakes may see main street mem bers tire sarah cham ber lain chief op er ing fin cial ficer republic main street part ner ship supports moderate gop lawmakers told national journal hop ing ry type ate comes huge mess sit ting two vocal gop critics conservative hardliners roiling house leadership suggested national journal thought retirement weighing members minds even currently considering stepping lot put hold ways people de cid ing run run rep pete king r ny told national journal saying personally considering retiring cause give likewise rep charlie dent r pa said pre par ing run ning reelec tion right see hap pens next two months go ing pretty intense dent said via talking points memo great news house gop moderates threaten may 

In [78]:
#converting the textual data to numerical data
vectorizer=TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

In [79]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7399359 stored elements and shape (44898, 115784)>
  Coords	Values
  (0, 5955)	0.08155087772812859
  (0, 8839)	0.036938253399384974
  (0, 9235)	0.09603322019311776
  (0, 9398)	0.21858586227433469
  (0, 9468)	0.03713336257978835
  (0, 14495)	0.040102312746254386
  (0, 15363)	0.056311551211951694
  (0, 15390)	0.04403028931410172
  (0, 16153)	0.11662945966414527
  (0, 16370)	0.06678203383651626
  (0, 16848)	0.033567878677933156
  (0, 17419)	0.11662945966414527
  (0, 17449)	0.10336974872009569
  (0, 18481)	0.09077874685695884
  (0, 19041)	0.03710733186304482
  (0, 19558)	0.04395278832634937
  (0, 19988)	0.03554945634363749
  (0, 20012)	0.0911411934517737
  (0, 21633)	0.04551930953493451
  (0, 22251)	0.042637103259311554
  (0, 23451)	0.09718398739429492
  (0, 24598)	0.14881712666994848
  (0, 29668)	0.04270551588277004
  (0, 31073)	0.07587806009817871
  (0, 31831)	0.024794123632794617
  :	:
  (44897, 80085)	0.03271643948482206
  (

In [80]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=2)

Training the model

In [81]:
model=LogisticRegression()

In [82]:
model.fit(X_train,Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


Evaluation

In [83]:
prediction=model.predict(X_train)
accuracy=accuracy_score(prediction,Y_train)

In [84]:
print('Accuracy score of the training data:',accuracy)

Accuracy score of the training data: 0.9931232251238933


In [85]:
test_prediction=model.predict(X_test)
test_accuracy=accuracy_score(test_prediction,Y_test)

In [86]:
print('Accuracy score of the test data:',test_accuracy)

Accuracy score of the test data: 0.9868596881959911


In [87]:
import pickle

# save model
with open("fake_news_model.pkl", "wb") as f:
    pickle.dump(model, f)

# save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)